# Grounding a Multi-Agent Orchestration with OpenAI Agents SDK and Bigdata: Financial Portfolio Analysis Example 

## Introduction

*This guide uses the OpenAI [Multi-Agent Orchestration](https://cookbook.openai.com/examples/agents_sdk/multi-agent-portfolio-collaboration/multi_agent_portfolio_collaboration)* cookbook to showcase how using Bigdata as a tool provides grounding capabilities to properly reference the sources used.

> **Note**
> 
> To a in depth explanation of the multi agent functionality please visit the origin Notebook from OpenAI linked above

**Basic description of the notebook**

In this notebook, the Bigdata grounding capabilities is leveraged inside a complex multi-agent collaboration system using OpenAI Agents SDK. 

In a nutshell, in this notebook:
- Multiple specialist agents are built (Macro, Fundamental, Quantitative) and collaborate under a Portfolio Manager agent to solve a challenging investment research problem.
- The Fundamental specialist agent has access to the **Bigdata** chat tool that provides grounded information.


---

## Table of Contents

1. [Grounding with Bigdata](#Grounding-with-Bigdata)
2. [Modifying the search tool](#Modifying-the-search-tool)
3. [Current dir setup](#Current-dir-set-up)
4. [Load Environment variables](#Load-environment-variables)
5. [Running the Workflow](#Running-the-Workflow)
6. [Example Output](#Example-Output)


---

## Grounding with Bigdata

In generative AI, grounding is the ability to connect AI models to verifiable sources of information. Grounding with Bigdata.com enhances your model’s accuracy and recency by letting you choose sources like web content, local, national, or regional news, official government sites, specialty content providers, and more. Along with factual answers, the Bigdata API provides inline supporting links (grounding sources), top search results, along with the response content.

To expand into Bigdata grounding please visit [Grounding with Bigdata](https://docs.bigdata.com/how-to-guides/search/grounding_with_bigdata)

We can benefit from grounding in the context of a Multi-Agent  of using the [BIgdata Chat Service](https://docs.bigdata.com/sdk-reference/chat/chatService)

---

## Modifying the search tool

When using the Agents SDK we can create **tools** that agents can use. Tools can range from simple Python functions to external services. In this project in particular we have modified one of the tools available to the Fundamental agent. It is as easy as decorating the function with the `@function_tool` decorator.

To expand on function tools [see the SDK docs.](https://openai.github.io/openai-agents-python/tools/#function-tools)

We have created two tools, once for searching news and another one for searching filings and transcripts.

### News search

In this tool we set up a time window of 6 months from today. We include source filters to retrieve the premium content data and use the `DocumentType.NEWS` scope. The function `run_search` handles conections and queries for use retrieving data. For each document retrieved we collect all the available chunks in a single text that will be fed into the Fundamental agent.

```python
@function_tool
def bigdata_search(user_request:str) -> str:
    """returns up to date information regarding the topic the user requests data for in terms of fundamental analysis. the output is a list of dictionaries containing text and the source"""
    print(f"Called bigdata_search with input: {user_request}")
    today = datetime.date.today()
    six_month_ago = today - relativedelta(months=6)
    
    # Format as YYYY-MM-DD
    today_str = today.strftime("%Y-%m-%d")
    six_month_ago_str = two_month_ago.strftime("%Y-%m-%d")
    
    this_month = AbsoluteDateRange(f"{six_month_ago_str}T08:00:00", f"{today_str}T00:00:00")
    
    sources_ids = []
    for sourcename in ["MT Newswires","Benzinga", "The Fly"]:
        source = bigdata.knowledge_graph.find_sources(sourcename)
        for id in source:
            sources_ids.append(id.id)
    
    query_news = Similarity(
        user_request
    ) & Any([Source(source) for source in tech_news_ids])
    
    results_news = run_search(bigdata=bigdata,
                         date_ranges = this_month,
                         queries=[query_news],
                         scope=DocumentType.NEWS,
                         sources=sources_ids,
                         limit=20)
    
    bigdata_response_news = []
    for i in results_news[0]:
        text = ""
        for j in i.chunks:
            text += j.text
        bigdata_response_news.append({'text': text, 'source': i.url})
    
    return json.dumps({'news': bigdata_response_news})
```

### Filings and transcripts search

In this tool we leverage the `TRANSCRIPTS` and `FILINGS` scopes from Bigdata. In this case we have defined the tool to accept both a `user_request`, that will guide the retrieval but also a `company_ticker`, that will ensure that we are retrieving information from the company we are asking about. On top of that we benefit from the `FiscalYear` filter, using current and past year.

```python
@function_tool
def bigdata_search_filings_transcripts(user_request:str,company_ticker:str) -> str:
    """returns up to date information regarding filings and transcripts for the given company ticker. the output is a list of dictionaries containing text and the source"""    

    print(f"Called bigdata_search_filings_transcripts with user_request: {user_request}\n and ticker: {company_ticker}")

    today = datetime.date.today()
    twelve_month_ago = today - relativedelta(months=12)
    
    # Format as YYYY-MM-DD
    today_str = today.strftime("%Y-%m-%d")
    twelve_month_ago_str = two_month_ago.strftime("%Y-%m-%d")
    
    this_month = AbsoluteDateRange(f"{twelve_month_ago_str}T08:00:00", f"{today_str}T00:00:00")
    
    ticker_id = bigdata.knowledge_graph.find_companies(company_ticker)[0].id
    query_transcripts = Similarity(user_request) & (FiscalYear(int(today.strftime("%Y"))) | FiscalYear(int(today.strftime("%Y"))-1)) & Entity(ticker_id)

    print(f"Bigdata id: {ticker_id}")

    results_transcripts = run_search(bigdata=bigdata,
                         date_ranges = this_month,
                         queries=[query_transcripts],
                         scope=DocumentType.TRANSCRIPTS,
                         limit=20)

    results_filings = run_search(bigdata=bigdata,
                     date_ranges = this_month,                                     
                     queries=[query_transcripts],
                     scope=DocumentType.FILINGS,
                     limit=20)
    
    bigdata_response_transcripts = []
    bigdata_response_filings = []

    for i in results_transcripts[0]:
        text = ""
        for j in i.chunks:
            text += j.text
        bigdata_response_transcripts.append({'text': text, 'source': i.headline})

    for i in results_filings[0]:
        text = ""
        for j in i.chunks:
            text += j.text
        bigdata_response_filings.append({'text': text, 'source': i.url})


    full_output = bigdata_response_transcripts + bigdata_response_filings
    random.shuffle(full_output)
    return json.dumps({'filings and transcripts': full_output})
```

---

## Current dir set up    

In [1]:
import os
import sys

current_dir = os.getcwd()

if current_dir not in sys.path:
    sys.path.append(current_dir)
print(f"✅ Local environment setup complete")

✅ Local environment setup complete


## Load environment variables

Mandatory environment variables:

- `BIGDATA_USERNAME`
- `BIGDATA_PASSWORD`
- `OPENAI_API_KEY`

The notebook is prepared for accessing data from FRED but it is not required, to use that functionality follow the guide:

- `FRED_API_KEY` (for FRED economic data, see [FRED API key instructions](https://fred.stlouisfed.org/docs/api/api_key.html))

In [2]:
import os
from dotenv import load_dotenv
from pathlib import Path

script_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
load_dotenv(script_dir / '.env')

BIGDATA_USERNAME = os.getenv('BIGDATA_USERNAME')
BIGDATA_PASSWORD = os.getenv('BIGDATA_PASSWORD')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not all([BIGDATA_USERNAME, BIGDATA_PASSWORD, OPENAI_API_KEY]):
    print("❌ Missing required environment variables")
    raise ValueError("Missing required environment variables. Check your .env file.")
else:
    print("✅ Credentials loaded from .env file")

print('Checking Bigdata client')

from tools import bigdata
bigdata


✅ Credentials loaded from .env file
Checking Bigdata client


In [3]:
try:
    import asyncio
    asyncio.get_running_loop()
    import nest_asyncio; nest_asyncio.apply()
    print("✅ nest_asyncio applied")
except (RuntimeError, ImportError):
    print("✅ nest_asyncio not needed")

✅ nest_asyncio applied


---

## Running the Workflow

Edit the question to whatever you'd like, but keep the date field to improve accuracy!

<div style="border-left: 4px solidrgb(0, 0, 0); padding: 0.5em; background:rgb(255, 229, 229);">
<strong>Disclaimer:</strong> This example is for educational purposes only. Consult a qualified financial professional before making any investment decisions
</div>


The workflow is kicked off by sending a user request to the Head Portfolio Manager (PM) agent. The PM agent orchestrates the entire process, delegating to specialist agents and tools as needed. You can monitor the workflow in real time using OpenAI Traces, which provide detailed visibility into every agent and tool call.

Edit the `question` in the code below to whatever you'd like, but keep the date field to improve accuracy! Ensure to reference the company ticker to boost data retrieval.

<div style="border-left: 4px solid #f39c12; padding: 0.5em; background: #fffbe6;">
<strong>Note:</strong> Depending on the complexity of the task, this request can take up to 10 minutes.
</div>


In [4]:
import datetime
import json
import os
from pathlib import Path
from contextlib import AsyncExitStack
from agents import Runner, add_trace_processor, trace
from agents.tracing.processors import BatchTraceProcessor
from utils import FileSpanExporter, output_file
from investment_agents.config import build_investment_agents
import asyncio

add_trace_processor(BatchTraceProcessor(FileSpanExporter()))

async def run_workflow():
    if "OPENAI_API_KEY" not in os.environ:
        raise EnvironmentError("OPENAI_API_KEY not set — set it as an environment variable before running.")

    today_str = datetime.date.today().strftime("%B %d, %Y")
    question = (
        f"Today is {today_str}. "
        "How would the planned interest rate reduction and board changes affect my holdings in GOOGL if they were to happen?"
        "Considering all the factors effecting its price right now (Macro, Technical, Fundamental, etc.), what is a realistic price target by the end of the year?"
    ) 
    question = (
        f"Today is {today_str}. "
        "With the US government now owning 10% of INTC, does this fundamentally change Intel's investment thesis from a struggling chipmaker to a strategic national asset - and what does that mean for long-term returns?"
    )     
    bundle = build_investment_agents()

    async with AsyncExitStack() as stack:
        for agent in [getattr(bundle, "fundamental", None), getattr(bundle, "quant", None)]:
            if agent is None:
                continue
            for server in getattr(agent, "mcp_servers", []):
                await server.connect()
                await stack.enter_async_context(server)

        print("Running multi-agent workflow with tracing enabled...\n")
        with trace(
            "Investment Research Workflow",
            metadata={"question": question[:512]}
        ) as workflow_trace:
            print(
                f"\n🔗 View the trace in the OpenAI console: "
                f"https://platform.openai.com/traces/trace?trace_id={workflow_trace.trace_id}\n"
            )

            response = None
            try:
                response = await asyncio.wait_for(
                    Runner.run(bundle.head_pm, question, max_turns=40),
                    timeout=1200
                )
            except asyncio.TimeoutError:
                print("\n❌ Workflow timed out after 20 minutes.")

            report_path = None
            try:
                if hasattr(response, 'final_output'):
                    output = response.final_output
                    if isinstance(output, str):
                        data = json.loads(output)
                        if isinstance(data, dict) and 'file' in data:
                            report_path = output_file(data['file'])
            except Exception as e:
                print(f"Could not parse investment report path: {e}")

            print(f"Workflow Completed Response from Agent: {response.final_output if hasattr(response, 'final_output') else response}, investment report created: {report_path if report_path else '[unknown]'}")
            return output

await run_workflow()

Running multi-agent workflow with tracing enabled...


🔗 View the trace in the OpenAI console: https://platform.openai.com/traces/trace?trace_id=trace_b1a3684ef73d4c98a0ef1f76ed35ca7f

Called bigdata_search with input: Impact of US government 10% ownership stake in Intel (INTC) on investment thesis, including national security, government support, political interference, regulatory scrutiny, capital allocation, and innovation. Recent analyst and expert commentary (2025).


Querying Bigdata...: 100%|████████████████████████| 1/1 [00:03<00:00,  3.08s/it]


Called bigdata_search_filings_transcripts with user_request: Management and board commentary on US government 10% stake, strategic direction, and implications for Intel's long-term returns and capital allocation (2025)
 and ticker: INTC
Bigdata id: 17EDA5


Querying Bigdata...: 100%|████████████████████████| 1/1 [00:04<00:00,  4.96s/it]


Called bigdata_search with input: US government 10% stake in Intel 2025 implications, US-China tech competition, global semiconductor supply chain, sector response


Querying Bigdata...: 100%|████████████████████████| 1/1 [00:01<00:00,  1.30s/it]


Workflow Completed Response from Agent: {"file": "investment_report.md"}, investment report created: /home/amartinezg/git/bigdata/github/bigdata-cookbook/Multi_Agent_Portfolio_Collaboration/outputs/investment_report.md


'{"file": "investment_report.md"}'


## Example Output

As shown in the example output below, the gounding has worked as expected. Now the fundamental analysis properly references all the news that are sumarized and analyzed.

Here is the example of the investment report generated through the workflow. The output is written to the `outputs` folder in the directory. 

<details>
    
# Investment Memo: Alphabet (GOOGL) – Impact of Planned Interest Rate Reduction, AI Strength, and Year-End 2025 Outlook

## Executive Summary

Alphabet (GOOGL) stands at the intersection of robust fundamental performance, a powerful AI narrative, and a complex macroeconomic backdrop. The investment thesis is that GOOGL remains a high-quality compounder, with its AI leadership and fortress balance sheet positioning it to benefit from a planned interest rate reduction. This thesis aligns with the firm's vision of seeking differentiated, evidence-based insights: while consensus is bullish, our synthesis highlights both the underappreciated risks (regulatory, macro volatility) and the potential for an "AI decoupling" scenario where GOOGL outperforms typical tech/macro patterns. The quant analysis provides original evidence that GOOGL's rate sensitivity is mild and not statistically significant, suggesting that the AI narrative and company-specific drivers may increasingly dominate price action. The best-case scenario sees GOOGL re-rated on AI execution and macro tailwinds, with price targets as high as \\$256. The worst-case scenario involves regulatory shocks or a sharp rotation out of tech, capping upside or triggering downside. Our recommendation is to maintain exposure but exercise vigilance, especially as the stock approaches the upper end of the target range without new positive catalysts. This approach embodies the firm's vision by planning for both best- and worst-case outcomes, challenging consensus where warranted, and grounding all views in evidence.

## Fundamentals Perspective

Alphabet is trading near all-time highs, recently closing at ~\\$209, with a 52-week range of \\$140.53–\\$210.52 ([GOOGL_6mo_1d_historical.csv](file)). The stock is valued at ~19x forward earnings, with consensus 2025 EPS estimates of \\$9.71–\\$10.20 and 2026–2027 EPS in the \\$10.20–\\$12.25 range ([bigdata_search](https://www.benzinga.com/node/46475467?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), [bigdata_search](https://www.benzinga.com/node/46638272?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)). Analyst price targets cluster around \\$212–\\$225, with some outliers as high as \\$234 ([bigdata_search](https://www.benzinga.com/node/46638272?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack), [GOOGL_recommendations_recommendations_f46b1fbe.csv](file)). Q2 2025 revenue was \\$96.4B (+14% YoY), net income \\$28.2B, and operating margins stable at 32% ([GOOGL_quarterly_income_stmt_8f6c3cf9.csv](file)). The balance sheet is exceptionally strong, with ~\\$95B in cash and robust free cash flow.

Alphabet’s business drivers—Search, YouTube, and Cloud—are all accelerating, with AI now a central pillar. Gemini LLMs are being rapidly deployed, with over 1.5B users of AI Overviews and strong enterprise adoption of Vertex AI and Gemini APIs ([Alphabet Inc: Q3 2024 Earnings Call](https://files.quartr.com/reports/38207-2025-07-24-10-53-02.pdf?ref=UmF2ZW5QYWNr)). Google’s full-stack AI approach (custom TPUs, data, models, and distribution) is a key moat. Management commentary highlights that AI-powered search monetizes at rates comparable to traditional search, and AI is driving higher engagement and click-through rates ([Alphabet Inc: Q1 2025 Earnings Call](https://files.quartr.com/reports/38207-2025-07-24-10-53-02.pdf?ref=UmF2ZW5QYWNr)). Cloud is growing at 32% YoY, with a \\$50B+ run rate, and the \\$85B 2025 CapEx plan is focused on AI infrastructure ([bigdata_search](https://www.benzinga.com/node/46638272?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)).

Catalysts include interest rate reductions, continued AI monetization, cloud growth, and capital allocation (buybacks, potential dividends). Regulatory and legal risks remain, but recent settlements have not materially impaired business momentum ([Alphabet 10-Q](https://www.sec.gov/Archives/edgar/data/1652044/000165204425000062/goog-20250630.htm)). Sell-side sentiment is strongly positive: 54/66 analysts rate GOOGL a Buy or Strong Buy ([GOOGL_recommendations_recommendations_f46b1fbe.csv](file)). However, risks include AI monetization lag, regulatory/antitrust action, high CapEx, competitive threats, and macro/FX volatility. The consensus expects stable double-digit revenue growth and robust AI monetization, but the variant view emphasizes either upside from faster AI monetization or downside from regulatory action or a structural shift in search behavior. The fundamental view is well-supported by recent financials and news, and aligns with consensus, but is differentiated by its explicit scenario planning and risk awareness ([GOOGL_news_c295c7a2.json](file)).

### Key Data Snapshots

**Recent Price History:**

| Date       |   Open |   High |    Low |   Close |   Volume |
|:-----------|-------:|-------:|-------:|--------:|---------:|
| 2025-08-13 | 204.13 | 204.53 | 197.51 |  201.96 | 28342900 |
| 2025-08-14 | 201.5  | 204.44 | 201.23 |  202.94 | 25230400 |
| 2025-08-15 | 203.85 | 206.44 | 201.28 |  203.9  | 34931400 |
| 2025-08-18 | 204.2  | 205.27 | 202.49 |  203.5  | 18526600 |
| 2025-08-19 | 203.03 | 203.44 | 199.96 |  201.57 | 24240200 |
| 2025-08-20 | 200.73 | 201.28 | 196.6  |  199.32 | 28955500 |
| 2025-08-21 | 199.75 | 202.48 | 199.43 |  199.75 | 19774600 |
| 2025-08-22 | 202.73 | 208.54 | 201.3  |  206.09 | 42827000 |
| 2025-08-25 | 206.43 | 210.52 | 205.28 |  208.49 | 29928900 |
| 2025-08-26 | 207.51 | 207.85 | 205.7  |  207.14 | 28447100 |

**Quarterly Financials (Q2 2025):**

| date       |   Net Income |   Total Revenue |   Operating Margin |
|:-----------|-------------:|----------------:|-------------------:|
| 2025-06-30 |   28.2B      |   96.4B         | 32%                |

**Analyst Recommendations:**

| period   |   strongBuy |   buy |   hold |   sell |   strongSell |
|:---------|------------:|------:|-------:|-------:|-------------:|
| 0m       |          13 |    41 |     12 |      0 |            0 |

## Macro Perspective

The macro environment is characterized by a Federal Reserve target rate of 4.25%–4.50%, inflation at 2.86% (core CPI 3.02%), and a softening labor market (unemployment at 4.2%). Political developments, including the dismissal of a Fed governor, have raised concerns about central bank independence and the potential for near-term rate cuts. The market is pricing in a steepening yield curve, with expectations of rate cuts followed by future inflationary pressures.

The consensus macro view is that rate cuts will support tech stock valuations by lowering discount rates and stimulating activity. However, the variant view—aligned with the firm's vision—emphasizes that the context of the rate cut (political interference, persistent inflation, economic weakness) could introduce volatility and offset the benefits. Tail risks include erosion of Fed credibility and persistent inflation, both of which could increase risk premiums and market volatility. For GOOGL, the net macro impact is likely positive but nuanced: lower rates are a tailwind, but the underlying reasons for the cuts and the risk of volatility must be monitored. This scenario planning is consistent with the firm's approach of not accepting consensus uncritically and planning for both best- and worst-case outcomes.

## Quantitative Perspective

The quantitative analysis provides original, evidence-based insights into GOOGL’s rate sensitivity and scenario outcomes. Regression analysis of GOOGL’s daily returns around FOMC event windows shows a coefficient of -0.0071 (p ≈ 0.086), indicating only a mild, statistically insignificant negative sensitivity to rate events:

|       Coef. |    Std.Err. |        t |     P>|t| |       [0.025 |     0.975] |
|------------:|------------:|---------:|----------:|-------------:|-----------:|
|  0.00140008 | 0.000847445 |  1.65212 | 0.0991413 | -0.000264931 | 0.00306509 |
| -0.00711127 | 0.00413511  | -1.71973 | 0.086103  | -0.0152357   | 0.00101314 |

Technical indicators show 30-day volatility at ~19–20% (annualized), 90-day volatility at ~27–33%, and RSI in the 66–76 range (bullish, but not overbought). MACD is positive, indicating upward momentum. Support is in the \\$174–\\$183 range, resistance at \\$203–\\$208.

| Date       |   Close |   vol_30d |   vol_90d |   RSI_14 |    MACD |   MACD_signal |   Support_30d |   Resistance_30d |
|:-----------|--------:|----------:|----------:|---------:|--------:|--------------:|--------------:|-----------------:|
| 2025-08-26 |  207.14 |  0.204914 |  0.272663 |  70.8412 | 4.9856  |       5.02716 |        182.97 |           208.49 |

Scenario analysis yields two year-end price targets:

| Scenario          |   Year-End Price | Assumptions                                                        |
|:------------------|-----------------:|:-------------------------------------------------------------------|
| Typical Tech/Rate |          235.38  | Rates fall, GOOGL follows tech/rate pattern, moderate upside       |
| AI Decoupling     |          256.09  | GOOGL decouples, AI strength, analyst bullish overlay (score=4.50) |

![GOOGL_price_volatility_technicals](outputs/GOOGL_price_volatility_technicals.png)

![GOOGL_year_end_scenarios](outputs/GOOGL_year_end_scenarios.png)

The quant view is differentiated by its scenario planning and by showing that GOOGL’s price is not tightly coupled to rate events, supporting the thesis that company-specific drivers (especially AI) may increasingly dominate. Limitations include the lack of direct FOMC time series (due to API constraints) and the qualitative nature of some macro overlays. Nonetheless, the analysis is robust and aligns with the firm vision by challenging consensus and planning for multiple outcomes.

## Portfolio Manager Perspective

The PM synthesis is that all three specialist sections converge on a bullish but nuanced outlook for GOOGL into year-end 2025. The consensus is that a planned interest rate reduction is a net positive, supporting higher multiples and potentially stimulating ad and IT spend. However, the macro section rightly highlights that the context of the rate cut—political interference, persistent inflation, or economic weakness—could introduce volatility and limit upside. The quant section’s scenario analysis is differentiated, showing that GOOGL’s price is only mildly rate-sensitive and that the AI narrative could drive a decoupling from typical macro patterns, with upside targets (\\$235–\\$256) above the fundamental consensus (\\$215–\\$225). The fundamental section, leveraging recent filings and news, confirms Alphabet’s AI strength and robust financials, but also flags regulatory and competitive risks. The PM’s pushback is that while the AI story is powerful, much of it is now priced in, and the risk of regulatory or macro shocks is underappreciated. The recommendation is to lean toward the lower end of the quant/fundamental target range (\\$215–\\$235) unless there is clear evidence of AI monetization inflecting or macro risks abating. Variant scenarios—such as a regulatory surprise or a sharp rotation out of tech—should not be ignored. Maintain vigilance and consider partial profit-taking if the stock approaches the upper end of the target range without new positive catalysts. This approach is fully aligned with the firm vision: it is differentiated, evidence-based, and plans for both best- and worst-case scenarios.

## Recommendation & Answer to the Question

Our recommendation is to maintain exposure to GOOGL, with a year-end 2025 price target range of \\$215–\\$235, reflecting both the consensus and variant scenario planning. This embodies the firm vision by grounding the thesis in original, evidence-based analysis, scenario planning, and a willingness to challenge consensus where warranted. The quant analysis supports the view that GOOGL’s price is not tightly coupled to rate events, and that the AI narrative could drive further upside if execution continues. However, much of the AI story is now priced in, and macro/regulatory risks are underappreciated. If the stock approaches the upper end of the target range without new positive catalysts, consider partial profit-taking. This recommendation is justified by the evidence across all three perspectives and is fully aligned with the firm’s differentiated, scenario-driven approach.

**END_OF_MEMO**



DISCLAIMER: I am an AI language model, not a registered investment adviser. Information provided is educational and general in nature. Consult a qualified financial professional before making any investment decisions.

</details>

In [ ]:
from bigdata_client import Bigdata
from bigdata_client.models.chat import ChatScope
from bigdata_client.models.chat import MarkdownLinkFormatter

from bigdata_research_tools.search.search import run_search

import datetime
from dateutil.relativedelta import relativedelta  # requires `python-dateutil` package
import random

from bigdata_client.query import Entity, Source, Any, Similarity, FiscalYear
from bigdata_client.daterange import AbsoluteDateRange
from bigdata_client.models.search import SortBy, DocumentType

bigdata = Bigdata()

today = datetime.date.today()
two_month_ago = today - relativedelta(months=12)

# Format as YYYY-MM-DD
today_str = today.strftime("%Y-%m-%d")
two_month_ago_str = two_month_ago.strftime("%Y-%m-%d")

this_month = AbsoluteDateRange(f"{two_month_ago_str}T08:00:00", f"{today_str}T00:00:00")

google_id = bigdata.knowledge_graph.find_companies('GOOGL')[0].id
user_request = "Recent analyst price targets and sentiment for GOOGL (Alphabet) as of August 2025"
query_transcripts = Similarity(user_request) & (FiscalYear(int(today.strftime("%Y"))) | FiscalYear(int(today.strftime("%Y"))-1)) & Entity(google_id)

search = bigdata.search.new(query_transcripts, scope=DocumentType.TRANSCRIPTS)
documents = search.run(1) 

for doc in documents:
    print(doc)